# Google Colab Setup

Run vehicle tracking and make/model census on Google Colab T4 GPU.

## 1. Get Car-Census code and data

In [ ]:
from google.colab import userdata
import os

# 1. Retrieve the secret securely
token = userdata.get('GITHUB_TOKEN')

# 2. Define the repo details
# Note: Remove 'https://' from the start of your repo string for the formatting below
repo_path = "github.com/DmitryMatv/Car-Census.git"
repo_url = f"https://{token}@{repo_path}"

# 3. Clone the repo
!git clone {repo_url}
    
# 4. Change directory to the cloned repo
os.chdir('Car-Census')

In [ ]:
# Option C: For private repos, authenticate with gh first
# !gh auth login
# !gh repo clone your-username/car-census
# import os
# os.chdir('car-census')

In [ ]:
# Option B: Upload via file picker
# from google.colab import files
# uploaded = files.upload()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r /content/drive/MyDrive/input_data /content/Car-Census/input_data/

## 2. Install Dependencies

In [ ]:
%cd /content/Car-Census

!pip install -e .
!pip uninstall -y onnxruntime
!pip install -U onnxruntime-gpu

## 3. Verify GPU, ONNX Runtime & FFmpeg

In [ ]:
import torch
import onnxruntime as ort

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
print(f"ONNX Runtime providers: {ort.get_available_providers()}")
!ffmpeg -hide_banner -encoders | grep nvenc || true

!ffmpeg -hide_banner -loglevel error \
  -f lavfi -i color=size=640x360:rate=1:duration=1 \
  -frames:v 1 -an -c:v h264_nvenc -f null -

In [ ]:
!ffmpeg -hide_banner -encoders | grep nvenc

## 4. Batch Tests

In [ ]:
import onnxruntime as ort

sess = ort.InferenceSession(
    "weights/yolo26m_fp16.onnx",
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
)

print("providers:", sess.get_providers())
for inp in sess.get_inputs():
    print("input:", inp.name, inp.shape, inp.type)
for out in sess.get_outputs():
    print("output:", out.name, out.shape, out.type)

In [ ]:
import time, subprocess, shlex

cmd = """
Car-Census analyze input_data/YtRoadTraffic_1440_30s.mp4 \
  --accelerator colab-t4 \
  --verbose
"""

start = time.perf_counter()
result = subprocess.run(shlex.split(cmd), text=True, capture_output=True)
elapsed = time.perf_counter() - start

print(result.stdout)
print(result.stderr)
print(f"elapsed_seconds={elapsed:.2f}")
print(f"video_seconds=30")
print(f"analysis_speed={30 / elapsed:.2f}x realtime")

In [ ]:
from config import build_effective_config
from pathlib import Path

config = build_effective_config(Path("."))
print("analysis.batch_size:", config.analysis.batch_size)
print("detector providers:", config.detector.onnx_execution_providers)

In [ ]:
from pathlib import Path
from config import build_effective_config
from detectors.factory import create_detector
from utils.video import iter_sampled_frames

config = build_effective_config(
    Path("."),
    overrides={
        "detector": {
            "onnx_execution_providers": ["CUDAExecutionProvider", "CPUExecutionProvider"],
            "onnx_require_gpu": True,
        }
    },
)

detector = create_detector(config, Path("."))

frames = []
for _, _, frame in iter_sampled_frames(
    Path("input_data/YtRoadTraffic_1440_30s.mp4"),
    source_fps=config.video.fps,
    target_fps=config.analysis.fps,
):
    frames.append(frame)
    if len(frames) == config.analysis.batch_size:
        break

print("requested batch:", len(frames))
print("onnx dynamic_batch:", getattr(detector, "dynamic_batch", None))
print("onnx fixed_batch_size:", getattr(detector, "fixed_batch_size", None))
print("onnx input_batch_dim:", getattr(detector, "input_batch_dim", None))

detections = detector.detect_batch(frames)
print("returned detection sets:", len(detections))
print("detections per frame:", [len(x) for x in detections])

## 4. Set Environment Variables

In [ ]:
traffic_eye_api_key = userdata.get('TRAFFICEYE_API_KEY')
if not traffic_eye_api_key:
    raise RuntimeError('Add TRAFFICEYE_API_KEY to Colab Secrets before running classification.')

os.environ['TRAFFICEYE_API_KEY'] = traffic_eye_api_key
print('TRAFFICEYE_API_KEY loaded from Colab Secrets.')

## 5. Run Car-Census

### Option A: Run Full Pipeline (detect, track, classify, render)

In [ ]:
!Car-Census run input_data/CarsHighwayTraffic_1440_10s.mp4 --accelerator colab-t4

### Option B: Run Without Classification (skip API calls)

In [ ]:
!Car-Census run input_data/YtRoadTraffic_1440_30s.mp4 --skip-classify --accelerator colab-t4

### Option C: Run with Camera ID

In [ ]:
!Car-Census run input_data/test4K.MP4 --camera-id my-camera --accelerator colab-t4

### Option D: ROI Edit (define polygon zone)

In [ ]:
!Car-Census roi edit input_data/test4K.MP4 --camera-id my-camera --device cuda

## 6. Download Results

In [ ]:
# Download output folder
from google.colab import files

!zip -r output.zip output/
files.download('output.zip')

In [ ]:
# Or download specific file
# files.download('/content/car-census/outputs/<run-id>/annotated.mp4')